In [161]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
import krippendorff
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import cohen_kappa_score
from statsmodels.stats.inter_rater import fleiss_kappa
import itertools
import re

In [162]:
df1 = pd.read_csv('text1.csv')
df2 = pd.read_csv('text2.csv')
df3 = pd.read_csv('text3.csv')

# Functions

## Prepairing

In [163]:
def prepare_for_kappa(subset):
    categories = sorted(set(subset.values.flatten()))
    result = []
    for _, row in subset.iterrows():
        counts = [list(row).count(cat) for cat in categories]
        result.append(counts)
    return pd.DataFrame(result, columns=categories)

In [164]:
def cols_for_version(df, version_tag="new"):
    return [c for c in df.columns if version_tag in c]

In [165]:
def parse_multilabel_cell(cell):
    """
    Converts a table cell containing labels into a list of labels.
    """
    if pd.isna(cell) or str(cell).strip() == "":
        return []
    if isinstance(cell, list):
        return [x.strip() for x in cell]
    parts = [x.strip() for x in re.split(r'[+,;]', str(cell)) if x.strip()]
    return parts


In [166]:
def make_binary_annotation(df, annotator_cols):
    """
    Converts annotations from multiple annotators into a binary matrix (multi-hot encoding).
    Each row represents an object, each column represents an annotator,
    and each cell contains a binary vector of all labels.
    """
    lists = df[annotator_cols].applymap(parse_multilabel_cell)

    all_labels = sorted(set(x for row in lists.values.flatten() for x in row))

    mlb = MultiLabelBinarizer(classes=all_labels)

    annot_matrices = []
    for col in annotator_cols:
        binarized = mlb.fit_transform(lists[col])
        annot_matrices.append(binarized)

    annot_array = np.stack(annot_matrices, axis=1)

    return {
        "matrix": annot_array,  
        "labels": all_labels,
        "annotators": annotator_cols,
    }

## Metrics

In [167]:
def mean_pairwise_jaccard(annot_array):
    """
    Calculates average pairwise Jaccard similarity between annotators.
    """
    n_samples, n_annotators, n_labels = annot_array.shape
    pairwise_vals = np.zeros((n_annotators, n_annotators))

    for i, j in itertools.combinations(range(n_annotators), 2):
        jaccard_per_sample = []
        for s in range(n_samples):
            a, b = annot_array[s, i, :], annot_array[s, j, :]
            inter = np.logical_and(a, b).sum()
            union = np.logical_or(a, b).sum()
            if union > 0:
                jaccard = inter / union
                jaccard_per_sample.append(jaccard)
        mean_jaccard = float(np.mean(jaccard_per_sample)) if jaccard_per_sample else 0.0
        pairwise_vals[i, j] = pairwise_vals[j, i] = mean_jaccard

    np.fill_diagonal(pairwise_vals, 1.0)
    mean_pairwise = float(np.mean(pairwise_vals[np.triu_indices(n_annotators, k=1)]))
    return mean_pairwise, pairwise_vals

In [168]:
def per_label_pairwise_kappa(annot_array, label_names, annotators):
    """
    Calculates mean pairwise Cohen's kappa for each label.
    """
    n_samples, n_annotators, n_labels = annot_array.shape
    results = []
    raw_kappa = {}

    for li, label in enumerate(label_names):
        kappas = []
        for i, j in itertools.combinations(range(n_annotators), 2):
            v1 = annot_array[:, i, li]
            v2 = annot_array[:, j, li]
            try:
                kappa = cohen_kappa_score(v1, v2)
                kappas.append(kappa)
                raw_kappa[(annotators[i], annotators[j], label)] = kappa
            except Exception:
                kappas.append(np.nan)
        results.append({
            "label": label,
            "mean_kappa": np.nanmean(kappas)
        })

    df_kappa = pd.DataFrame(results).sort_values("mean_kappa", ascending=False).reset_index(drop=True)
    return df_kappa, raw_kappa

In [169]:
def krippendorff_alpha_per_label(annot_array, label_names):
    """
    Calculates Krippendorff's alpha for each label.
    """
    results = []
    alphas = {}

    for li, label in enumerate(label_names):
        data = annot_array[:, :, li].T
        try:
            alpha = krippendorff.alpha(reliability_data=data, level_of_measurement='nominal')
            alphas[label] = alpha
            results.append({"label": label, "alpha": alpha})
        except Exception:
            alphas[label] = np.nan
            results.append({"label": label, "alpha": np.nan})

    df_alpha = pd.DataFrame(results).sort_values("alpha", ascending=False).reset_index(drop=True)
    mean_alpha = float(np.nanmean(list(alphas.values())))
    return df_alpha, mean_alpha

## Main pipeline

In [170]:
def analyze_version(df, version_tag, top_n=10):
    annotator_cols = [c for c in df.columns if version_tag in c]

    annot_bin = make_binary_annotation(df, annotator_cols)
    label_names = annot_bin["labels"]
    binary_matrix = annot_bin["matrix"]
    mean_j, jaccard_df = mean_pairwise_jaccard(binary_matrix)
    print(f"\nMean pairwise Jaccard: {mean_j:.4f}")

    df_kappa, raw_kappa = per_label_pairwise_kappa(
        annot_bin["matrix"], 
        label_names, 
        annot_bin["annotators"]
    )
    print(f"\nTop-{top_n} labels by mean pairwise Cohen's kappa:")
    print(df_kappa.head(top_n).to_string(index=False))
    
    df_alpha, mean_alpha = krippendorff_alpha_per_label(annot_bin["matrix"], label_names)    
    print(f"\nMean Krippendorff's alpha: {mean_alpha:.4f}")
    print(f"\nTop-{top_n} labels by Krippendorff's alpha:")
    print(df_alpha.head(top_n).to_string(index=False))
    
    return {
        "label_names": label_names,
        "mean_pairwise_jaccard": mean_j,
        "pairwise_jaccard_df": jaccard_df,
        "per_label_kappa_df": df_kappa,
        "per_label_kappa_raw": raw_kappa,
        "krippendorff_df": df_alpha,
        "mean_krippendorff": mean_alpha
    }

# Text 1

## Without missing cells

In [171]:
df1 = pd.read_csv('text1.csv')
df1.drop(columns=['Sent', 'annotator_new1'], inplace=True, errors='ignore')
df1.dropna(inplace=True)
df1

,Error,annotator_new2,annotator_new3,annotator_new4,annotator_old1,annotator_old2,annotator_old3
0,прожиить,Ortho,Ortho,Ortho,Ortho,Ortho,Ortho + Extra
1,перейтии,Ortho,Ortho,Ortho,Ortho,Ortho,Ortho + Extra
2,влияют нашу жизнь,Miss + Gov + Prep,Prep + Miss,Prep + Miss,ArgStr,Gov + Miss + Prep,Prep + Miss
3,каждий ошибка,Gender,AgrNum + Ortho,AgrGender + Ortho +,AgrGender + Ortho,AgrGender,Ortho + Mode + AgrGender
4,считается как грех,Extra + Lex,Conj + Gov,Constr,ArgStr,Gov,Conj + Extra
5,рабочные,Ortho,Lex,Ortho,Deriv,Deriv + Insert,Deriv
7,били,Ortho,Ortho,Ortho,Ortho,Ortho,Ortho
8,идеальними,Ortho,Ortho,Ortho,Ortho,Ortho + Extra,Ortho + Extra
9,польними,Ortho,Ortho,Ortho,Ortho,Ortho + Extra,Ortho + Extra
10,чувствуем грусть и депрессию,Lex + Miss,Lex,Lex,Not-clear,Constr + Lex + Miss,Idiom + Miss + Transp


In [172]:
new1 = [c for c in df1.columns if "new" in c]
old1 = [c for c in df1.columns if "old" in c]

new_df = prepare_for_kappa(df1[new1])
old_df = prepare_for_kappa(df1[old1])

kappa_new = fleiss_kappa(new_df.values)
kappa_old = fleiss_kappa(old_df.values)

print(f"Fleiss' kappa (new): {kappa_new:.4f}")
print(f"Fleiss' kappa (old): {kappa_old:.4f}")

Fleiss' kappa (new): 0.3520
Fleiss' kappa (old): 0.2283


In [173]:
res_new = analyze_version(df1, version_tag="new", top_n=10)
res_old = analyze_version(df1, version_tag="old", top_n=10)
print(f"Mean pairwise Jaccard: new = {res_new['mean_pairwise_jaccard']:.4f}, old = {res_old['mean_pairwise_jaccard']:.4f}")
print(f"Mean Krippendorff alpha: new = {res_new['mean_krippendorff']:.4f}, old = {res_old['mean_krippendorff']:.4f}")


Mean pairwise Jaccard: 0.5286

Top-10 labels by mean pairwise Cohen's kappa:
  label  mean_kappa
  Extra    0.677575
    Ref    0.664061
AgrPers    0.637123
  Ortho    0.597032
   Miss    0.492978
   Prep    0.466501
    Com    0.333333
    Gov    0.321115
   Conj    0.304348
    Lex    0.280174

Mean Krippendorff's alpha: 0.2579

Top-10 labels by Krippendorff's alpha:
  label    alpha
  Extra 0.687243
    Ref 0.668605
AgrPers 0.648148
  Ortho 0.599532
    Com 0.494681
   Miss 0.472222
   Prep 0.460227
    Gov 0.340278
    Asp 0.318996
    Lex 0.295644

Mean pairwise Jaccard: 0.4457

Top-10 labels by mean pairwise Cohen's kappa:
    label  mean_kappa
      Com    1.000000
AgrGender    1.000000
    Ortho    0.907825
   Transp    0.855856
      Lex    0.768116
    Deriv    0.768116
    Extra    0.447120
  AgrPers    0.333333
     Prep    0.261261
     Miss    0.261261

Mean Krippendorff's alpha: 0.2786

Top-10 labels by Krippendorff's alpha:
    label    alpha
AgrGender 1.000000
      C

/var/folders/v3/g9p917f512s52x0xb8r6htn00000gn/T/ipykernel_59428/3735522103.py:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  lists = df[annotator_cols].applymap(parse_multilabel_cell)
/Users/mac/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Users/mac/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:897: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/Users/mac/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Users/mac/m

## With missing cells

In [174]:
df1 = pd.read_csv('text1.csv')
df1.drop(columns=['Sent', 'annotator_new1'], inplace=True, errors='ignore')
df1.fillna('N/A', inplace=True)
df1

,Error,annotator_new2,annotator_new3,annotator_new4,annotator_old1,annotator_old2,annotator_old3
0,прожиить,Ortho,Ortho,Ortho,Ortho,Ortho,Ortho + Extra
1,перейтии,Ortho,Ortho,Ortho,Ortho,Ortho,Ortho + Extra
2,влияют нашу жизнь,Miss + Gov + Prep,Prep + Miss,Prep + Miss,ArgStr,Gov + Miss + Prep,Prep + Miss
3,каждий ошибка,Gender,AgrNum + Ortho,AgrGender + Ortho +,AgrGender + Ortho,AgrGender,Ortho + Mode + AgrGender
4,считается как грех,Extra + Lex,Conj + Gov,Constr,ArgStr,Gov,Conj + Extra
5,рабочные,Ortho,Lex,Ortho,Deriv,Deriv + Insert,Deriv
6,идеальные,Gov,AgrCase,AgrCase,Gov,Brev,N/A
7,били,Ortho,Ortho,Ortho,Ortho,Ortho,Ortho
8,идеальними,Ortho,Ortho,Ortho,Ortho,Ortho + Extra,Ortho + Extra
9,польними,Ortho,Ortho,Ortho,Ortho,Ortho + Extra,Ortho + Extra


In [175]:
new1 = [c for c in df1.columns if "new" in c]
old1 = [c for c in df1.columns if "old" in c]

new_df = prepare_for_kappa(df1[new1])
old_df = prepare_for_kappa(df1[old1])

kappa_new = fleiss_kappa(new_df.values)
kappa_old = fleiss_kappa(old_df.values)

print(f"Fleiss' kappa (new): {kappa_new:.4f}")
print(f"Fleiss' kappa (old): {kappa_old:.4f}")

Fleiss' kappa (new): 0.3524
Fleiss' kappa (old): 0.2209


In [176]:
res_new = analyze_version(df1, version_tag="new", top_n=10)
res_old = analyze_version(df1, version_tag="old", top_n=10)
print(f"Mean pairwise Jaccard: new = {res_new['mean_pairwise_jaccard']:.4f}, old = {res_old['mean_pairwise_jaccard']:.4f}")
print(f"Mean Krippendorff alpha: new = {res_new['mean_krippendorff']:.4f}, old = {res_old['mean_krippendorff']:.4f}")


Mean pairwise Jaccard: 0.5227

Top-10 labels by mean pairwise Cohen's kappa:
  label  mean_kappa
  Extra    0.679263
    Ref    0.665217
AgrPers    0.637712
  Ortho    0.603656
   Miss    0.493921
   Prep    0.467974
AgrCase    0.333333
    Com    0.333333
   Conj    0.305263
    Lex    0.283252

Mean Krippendorff's alpha: 0.2681

Top-10 labels by Krippendorff's alpha:
  label    alpha
  Extra 0.688889
    Ref 0.669663
AgrPers 0.648746
  Ortho 0.606250
AgrCase 0.494845
    Com 0.494845
   Miss 0.473118
   Prep 0.461538
    Asp 0.319444
    Lex 0.298748

Mean pairwise Jaccard: 0.4322

Top-10 labels by mean pairwise Cohen's kappa:
    label  mean_kappa
      Com    1.000000
AgrGender    1.000000
    Ortho    0.909254
   Transp    0.856209
      Lex    0.768421
    Deriv    0.768421
    Extra    0.451873
  AgrPers    0.333333
     Miss    0.261438
     Prep    0.261438

Mean Krippendorff's alpha: 0.2586

Top-10 labels by Krippendorff's alpha:
    label    alpha
AgrGender 1.000000
      C

/var/folders/v3/g9p917f512s52x0xb8r6htn00000gn/T/ipykernel_59428/3735522103.py:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  lists = df[annotator_cols].applymap(parse_multilabel_cell)
/Users/mac/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Users/mac/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:897: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/Users/mac/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Users/mac/m

# Text 2

## With missing cells

In [177]:
df2 = pd.read_csv('text2.csv')
df2.drop(columns=['Sent'], inplace=True, errors='ignore')
df2.fillna('N/A', inplace=True)
df2

,Error,annotator_new1,annotator_new2,annotator_new3,annotator_old1,annotator_old2,annotator_old3
0,тем больше мне нравятся все новые места и города,Ref + Extra + Transfer,N/A,Lex + Extra,Not-clear,Lex,Ref + Extra
1,дестинации,Gov + Lex + Transfer,CS + Gov,Gov,CS,Transfer,CS
2,влюблюсь,Asp,Asp,Tense,Asp,Asp,Asp
3,в его,Ref + AgrNum,Ref,Gov + Ref,Ortho,Ref,Ref + Subst
4,узнавать,Lex + Transfer,Lex,Lex,Asp,Lex,Lex
5,по данном городе,Gov\r\n,Gov + AgrCase,Gov,ArgStr,AgrCase + AgrGender,Prep + Subst
6,окунуться,Asp,Asp,Asp,Asp,Asp,Lex
7,мы не возможны,Lex,Lex,Lex,Asp,Morph + Constr + Syntax,Mode
8,чувствую себя очень грустно,Idiom + Transfer,Constr,Constr,Constr,Сonstr,Lex
9,мне привлекают,Gov,Ref,Ref,Gov,Gov + Constr,Ref + Subst


In [178]:
new2 = [c for c in df2.columns if "new" in c]
old2 = [c for c in df2.columns if "old" in c]

new_df = prepare_for_kappa(df2[new2])
old_df = prepare_for_kappa(df2[old2])

kappa_new = fleiss_kappa(new_df.values)
kappa_old = fleiss_kappa(old_df.values)

print(f"Fleiss' kappa (new): {kappa_new:.4f}")
print(f"Fleiss' kappa (old): {kappa_old:.4f}")

Fleiss' kappa (new): 0.3554
Fleiss' kappa (old): 0.1471


In [179]:
res_new = analyze_version(df2, version_tag="new", top_n=10)
res_old = analyze_version(df2, version_tag="old", top_n=10)
print(f"Mean pairwise Jaccard: new = {res_new['mean_pairwise_jaccard']:.4f}, old = {res_old['mean_pairwise_jaccard']:.4f}")
print(f"Mean Krippendorff alpha: new = {res_new['mean_krippendorff']:.4f}, old = {res_old['mean_krippendorff']:.4f}")


Mean pairwise Jaccard: 0.5082

Top-10 labels by mean pairwise Cohen's kappa:
 label  mean_kappa
    WO    1.000000
   Lex    0.785714
   Asp    0.633476
   Gov    0.480688
   Ref    0.468354
Constr    0.352468
  Miss    0.333333
 Voice    0.333333
 Extra    0.223239
 Ortho    0.216450

Mean Krippendorff's alpha: 0.2473

Top-10 labels by Krippendorff's alpha:
 label    alpha
    WO 1.000000
   Lex 0.788360
   Asp 0.644444
  Miss 0.493671
 Voice 0.493671
   Ref 0.485294
   Gov 0.457014
Constr 0.368421
 Ortho 0.316239
 Extra 0.288889

Mean pairwise Jaccard: 0.2469

Top-10 labels by mean pairwise Cohen's kappa:
 label  mean_kappa
   Asp    0.547231
Altern    0.333333
    CS    0.333333
   Gov    0.288998
   Lex    0.265794
    WO    0.216450
  Prep    0.156863
 Extra    0.120735
   Ref    0.030812
 Morph    0.000000

Mean Krippendorff's alpha: 0.0872

Top-10 labels by Krippendorff's alpha:
  label    alpha
    Asp 0.536680
 Altern 0.493671
     CS 0.493671
    Gov 0.382716
     WO 0.31623

/var/folders/v3/g9p917f512s52x0xb8r6htn00000gn/T/ipykernel_59428/3735522103.py:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  lists = df[annotator_cols].applymap(parse_multilabel_cell)
/Users/mac/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Users/mac/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:897: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/Users/mac/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Users/mac/m

## Without missing cells

In [180]:
df2 = pd.read_csv('text2.csv')
df2.drop(columns=['Sent'], inplace=True, errors='ignore')
df2.dropna(inplace=True)
df2

,Error,annotator_new1,annotator_new2,annotator_new3,annotator_old1,annotator_old2,annotator_old3
1,дестинации,Gov + Lex + Transfer,CS + Gov,Gov,CS,Transfer,CS
2,влюблюсь,Asp,Asp,Tense,Asp,Asp,Asp
3,в его,Ref + AgrNum,Ref,Gov + Ref,Ortho,Ref,Ref + Subst
4,узнавать,Lex + Transfer,Lex,Lex,Asp,Lex,Lex
5,по данном городе,Gov\r\n,Gov + AgrCase,Gov,ArgStr,AgrCase + AgrGender,Prep + Subst
6,окунуться,Asp,Asp,Asp,Asp,Asp,Lex
7,мы не возможны,Lex,Lex,Lex,Asp,Morph + Constr + Syntax,Mode
8,чувствую себя очень грустно,Idiom + Transfer,Constr,Constr,Constr,Сonstr,Lex
9,мне привлекают,Gov,Ref,Ref,Gov,Gov + Constr,Ref + Subst
10,сказать,Lex,Lex,Lex,Lex,Lex,Lex


In [181]:
new2 = [c for c in df2.columns if "new" in c]
old2 = [c for c in df2.columns if "old" in c]

new_df = prepare_for_kappa(df2[new2])
old_df = prepare_for_kappa(df2[old2])

kappa_new = fleiss_kappa(new_df.values)
kappa_old = fleiss_kappa(old_df.values)

print(f"Fleiss' kappa (new): {kappa_new:.4f}")
print(f"Fleiss' kappa (old): {kappa_old:.4f}")

Fleiss' kappa (new): 0.3733
Fleiss' kappa (old): 0.1808


In [182]:
res_new = analyze_version(df2, version_tag="new")
res_old = analyze_version(df2, version_tag="old")
print(f"Mean pairwise Jaccard: new = {res_new['mean_pairwise_jaccard']:.4f}, old = {res_old['mean_pairwise_jaccard']:.4f}")
print(f"Mean Krippendorff alpha: new = {res_new['mean_krippendorff']:.4f}, old = {res_old['mean_krippendorff']:.4f}")


Mean pairwise Jaccard: 0.5495

Top-10 labels by mean pairwise Cohen's kappa:
 label  mean_kappa
    WO    1.000000
   Lex    0.845389
   Asp    0.629323
   Gov    0.509728
   Ref    0.494505
Constr    0.345665
  Miss    0.333333
 Extra    0.303030
 Ortho    0.215385
 Tense    0.000000

Mean Krippendorff's alpha: 0.2583

Top-10 labels by Krippendorff's alpha:
 label    alpha
    WO 1.000000
   Lex 0.846154
   Asp 0.640212
   Ref 0.518868
  Miss 0.492537
   Gov 0.467085
Constr 0.362500
 Extra 0.313131
 Ortho 0.313131
 Tense 0.000000

Mean pairwise Jaccard: 0.2754

Top-10 labels by mean pairwise Cohen's kappa:
  label  mean_kappa
    Asp    0.541415
    Lex    0.337808
 Altern    0.333333
     CS    0.333333
    Gov    0.333333
     WO    0.215385
  Extra    0.215385
    Ref    0.033520
AgrCase    0.000000
  Ortho    0.000000

Mean Krippendorff's alpha: 0.0961

Top-10 labels by Krippendorff's alpha:
    label    alpha
      Asp 0.529954
   Altern 0.492537
       CS 0.492537
      Gov 0.4

/var/folders/v3/g9p917f512s52x0xb8r6htn00000gn/T/ipykernel_59428/3735522103.py:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  lists = df[annotator_cols].applymap(parse_multilabel_cell)
/Users/mac/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Users/mac/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:897: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/Users/mac/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Users/mac/m

# Text 3

## Without missing cells

In [183]:
df3 = pd.read_csv('text3.csv')
df3.drop(columns=['Sent'], inplace=True, errors='ignore')
df3.dropna(inplace=True)
df3

,Error,annotator_new1,annotator_new2,annotator_new3,annotator_old1,annotator_old2,annotator_old3
3,возникли у него,WO,Lex,WO,Num,Lex,Lex
4,отношения,Num + Lex + Transfer,Lex,Num + Miss,Lex,Num,Lex
7,воспоминание,Num + Ref + Miss,Constr,Num,Num + WO,Num,Num
8,соединенные,Brev,Brev,Voice,Brev,AgrCase,Brev
9,вырубленные,Brev,Brev,Voice,Brev,Brev,Brev
12,это дерево из сибирской лиственницы,Syntax + Constr,Lex+Extra,Constr,Lex,Lex,Lex + Prep + Extra
18,несколько,Aux + Miss,Lex,Lex,Lex,Miss + Constr,Aux + Miss
19,возможностей,Lex,Lex,Lex,Lex,Lex,Lex
20,двигаться,Lex,Lex,Lex,Lex,Lex,Lex + Asp + Subst
21,корабль,Aux + Miss,Aux + Lex,Lex + Extra,Lex,Lex,Aux + Miss


In [184]:
new3 = [c for c in df3.columns if "new" in c]
old3 = [c for c in df3.columns if "old" in c]

new_df = prepare_for_kappa(df3[new3])
old_df = prepare_for_kappa(df3[old3])

kappa_new = fleiss_kappa(new_df.values)
kappa_old = fleiss_kappa(old_df.values)

print(f"Fleiss' kappa (new): {kappa_new:.4f}")
print(f"Fleiss' kappa (old): {kappa_old:.4f}")

Fleiss' kappa (new): 0.3511
Fleiss' kappa (old): 0.2105


In [185]:
res_new = analyze_version(df3, version_tag="new")
res_old = analyze_version(df3, version_tag="old")
print(f"Mean pairwise Jaccard: new = {res_new['mean_pairwise_jaccard']:.4f}, old = {res_old['mean_pairwise_jaccard']:.4f}")
print(f"Mean Krippendorff alpha: new = {res_new['mean_krippendorff']:.4f}, old = {res_old['mean_krippendorff']:.4f}")


Mean pairwise Jaccard: 0.4846

Top-10 labels by mean pairwise Cohen's kappa:
 label  mean_kappa
 Tense    1.000000
   Gov    0.527317
   Num    0.476977
   Lex    0.450958
    WO    0.420080
  Infl    0.333333
  Conj    0.333333
  Brev    0.333333
   Asp    0.333333
Constr    0.220588

Mean Krippendorff's alpha: 0.2789

Top-10 labels by Krippendorff's alpha:
label    alpha
Tense 1.000000
  Gov 0.536680
  Num 0.506173
 Infl 0.493671
  Asp 0.493671
 Conj 0.493671
   WO 0.480519
 Brev 0.466667
  Lex 0.453416
  Aux 0.316239

Mean pairwise Jaccard: 0.4372

Top-10 labels by mean pairwise Cohen's kappa:
  label  mean_kappa
  Tense    1.000000
   Brev    0.573425
    Lex    0.459800
    Num    0.421140
   Conj    0.333333
   Miss    0.210046
AgrCase    0.156863
 Gender    0.000000
  Subst    0.000000
    Ref    0.000000

Mean Krippendorff's alpha: 0.1384

Top-10 labels by Krippendorff's alpha:
    label    alpha
    Tense 1.000000
     Brev 0.589041
     Conj 0.493671
      Lex 0.463415
     

/var/folders/v3/g9p917f512s52x0xb8r6htn00000gn/T/ipykernel_59428/3735522103.py:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  lists = df[annotator_cols].applymap(parse_multilabel_cell)
/Users/mac/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Users/mac/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:897: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/Users/mac/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Users/mac/m

## With missing cells

In [186]:
df3 = pd.read_csv('text3.csv')
df3.drop(columns=['Sent'], inplace=True, errors='ignore')
df3.fillna('N/A', inplace=True)
df3

,Error,annotator_new1,annotator_new2,annotator_new3,annotator_old1,annotator_old2,annotator_old3
0,понимать,Lex + Transfer,Lex,Lex,N/A,Lex,Lex
1,взгляду,Constr + Transfer,Lex,Lex,N/A,Constr,N/A
2,отношениях,Gov + Lex + Transfer + Ref + Miss,Lex,Gov,N/A,Syntax,AgrCase + Num
3,возникли у него,WO,Lex,WO,Num,Lex,Lex
4,отношения,Num + Lex + Transfer,Lex,Num + Miss,Lex,Num,Lex
5,к,Prep,N/A,N/A,N/A,N/A,Prep + Subst
6,месту,Gov + Ref + Miss,N/A,N/A,N/A,Ref,Case
7,воспоминание,Num + Ref + Miss,Constr,Num,Num + WO,Num,Num
8,соединенные,Brev,Brev,Voice,Brev,AgrCase,Brev
9,вырубленные,Brev,Brev,Voice,Brev,Brev,Brev


In [187]:
new3 = [c for c in df3.columns if "new" in c]
old3 = [c for c in df3.columns if "old" in c]

new_df = prepare_for_kappa(df3[new3])
old_df = prepare_for_kappa(df3[old3])

kappa_new = fleiss_kappa(new_df.values)
kappa_old = fleiss_kappa(old_df.values)

print(f"Fleiss' kappa (new): {kappa_new:.4f}")
print(f"Fleiss' kappa (old): {kappa_old:.4f}")

Fleiss' kappa (new): 0.2545
Fleiss' kappa (old): 0.1463


In [188]:
res_new = analyze_version(df3, version_tag="new")
res_old = analyze_version(df3, version_tag="old")
print(f"Mean pairwise Jaccard: new = {res_new['mean_pairwise_jaccard']:.4f}, old = {res_old['mean_pairwise_jaccard']:.4f}")
print(f"Mean Krippendorff alpha: new = {res_new['mean_krippendorff']:.4f}, old = {res_old['mean_krippendorff']:.4f}")


Mean pairwise Jaccard: 0.3911

Top-10 labels by mean pairwise Cohen's kappa:
 label  mean_kappa
 Tense    1.000000
   Num    0.494818
   Gov    0.456017
    WO    0.430710
   Lex    0.361608
  Infl    0.333333
  Brev    0.281967
Constr    0.232838
AgrNum    0.218978
   Asp    0.218978

Mean Krippendorff's alpha: 0.2429

Top-10 labels by Krippendorff's alpha:
 label    alpha
 Tense 1.000000
   Num 0.528620
  Infl 0.496403
    WO 0.489051
   Gov 0.465649
  Brev 0.402985
   Lex 0.360199
   Aux 0.323671
   Asp 0.323671
AgrNum 0.323671

Mean pairwise Jaccard: 0.3223

Top-10 labels by mean pairwise Cohen's kappa:
  label  mean_kappa
  Tense    0.771290
   Brev    0.546575
    Num    0.388387
    Lex    0.329642
   Conj    0.281967
   Miss    0.135021
AgrCase    0.097744
    N/A    0.083808
    Gov    0.000000
     WO    0.000000

Mean Krippendorff's alpha: 0.0825

Top-10 labels by Krippendorff's alpha:
       label    alpha
       Tense 0.744526
        Brev 0.572519
        Conj 0.402985
 

/var/folders/v3/g9p917f512s52x0xb8r6htn00000gn/T/ipykernel_59428/3735522103.py:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  lists = df[annotator_cols].applymap(parse_multilabel_cell)
/Users/mac/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Users/mac/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:897: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/Users/mac/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Users/mac/m

# Combined calculation

In [189]:
df1.drop(columns=['Sent', 'annotator_new1', 'annotator_new2', 'annotator_old2'], inplace=True, errors='ignore')
df1.rename(columns={'annotator_new3': 'annotator_new1', 'annotator_new4': 'annotator_new2', 'annotator_old3': 'annotator_old2'}, inplace=True)
df2.drop(columns=['Sent', 'annotator_new1', 'annotator_old2'], inplace=True, errors='ignore')
df2.rename(columns={'annotator_new2': 'annotator_new1', 'annotator_new3': 'annotator_new2', 'annotator_old3': 'annotator_old2'}, inplace=True)
df3.drop(columns=['Sent', 'annotator_new1', 'annotator_old2'], inplace=True, errors='ignore')
df3.rename(columns={'annotator_new2': 'annotator_new1', 'annotator_new3': 'annotator_new2', 'annotator_old3': 'annotator_old2'}, inplace=True)
all_df = pd.concat([df1, df2, df3], ignore_index=True)
all_df.fillna('N/A', inplace=True)
# all_df = all_df.dropna()
all_df

,Error,annotator_new1,annotator_new2,annotator_old1,annotator_old2
0,прожиить,Ortho,Ortho,Ortho,Ortho + Extra
1,перейтии,Ortho,Ortho,Ortho,Ortho + Extra
2,влияют нашу жизнь,Prep + Miss,Prep + Miss,ArgStr,Prep + Miss
3,каждий ошибка,AgrNum + Ortho,AgrGender + Ortho +,AgrGender + Ortho,Ortho + Mode + AgrGender
4,считается как грех,Conj + Gov,Constr,ArgStr,Conj + Extra
...,...,...,...,...,...
98,что,Conj,N/A,N/A,Conj + Miss
99,Венеция,Gov,Gov,Constr,Case
100,это производит грустное впечатление,Constr,Constr + Lex,Constr,Disc + Lex + Num
101,воображение,Lex,Lex,Lex,Lex + Num


In [190]:
kappa_new = cohen_kappa_score(all_df["annotator_new1"], all_df["annotator_new2"])

kappa_old = cohen_kappa_score(all_df["annotator_old1"], all_df["annotator_old2"])

print(f"Cohen's kappa (new): {kappa_new:.4f}")
print(f"Cohen's kappa (old): {kappa_old:.4f}")

Cohen's kappa (new): 0.4036
Cohen's kappa (old): 0.1745


In [191]:
res_new = analyze_version(all_df, version_tag="new")
res_old = analyze_version(all_df, version_tag="old")
print(f"Mean pairwise Jaccard: new = {res_new['mean_pairwise_jaccard']:.4f}, old = {res_old['mean_pairwise_jaccard']:.4f}")
print(f"Mean Krippendorff alpha: new = {res_new['mean_krippendorff']:.4f}, old = {res_old['mean_krippendorff']:.4f}")


Mean pairwise Jaccard: 0.5097

Top-10 labels by mean pairwise Cohen's kappa:
  label  mean_kappa
    Ref    0.826111
  Ortho    0.684740
AgrCase    0.662295
AgrPers    0.662295
     WO    0.656667
    Lex    0.575212
  Extra    0.425498
    Gov    0.424581
  Tense    0.390533
    Num    0.390533

Mean Krippendorff's alpha: 0.2600

Top-10 labels by Krippendorff's alpha:
  label    alpha
    Ref 0.826907
  Ortho 0.682873
AgrCase 0.663383
AgrPers 0.663383
     WO 0.658333
    Lex 0.575014
  Extra 0.428059
    Gov 0.425770
    Num 0.388060
  Tense 0.388060

Mean pairwise Jaccard: 0.2896

Top-10 labels by mean pairwise Cohen's kappa:
    label  mean_kappa
AgrGender    1.000000
      Com    1.000000
    Ortho    0.863456
     Brev    0.740554
    Tense    0.662295
       CS    0.662295
   Transp    0.657807
      Num    0.374494
      Lex    0.343639
    Deriv    0.322368

Mean Krippendorff's alpha: 0.1783

Top-10 labels by Krippendorff's alpha:
    label    alpha
AgrGender 1.000000
      C

/var/folders/v3/g9p917f512s52x0xb8r6htn00000gn/T/ipykernel_59428/3735522103.py:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  lists = df[annotator_cols].applymap(parse_multilabel_cell)
/var/folders/v3/g9p917f512s52x0xb8r6htn00000gn/T/ipykernel_59428/3735522103.py:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  lists = df[annotator_cols].applymap(parse_multilabel_cell)


# TO DO:
- считать по одному тегу (если встречается хотя бы один раз, то строчку берём в анализ)
- считать внутри гипертега
- посчитать как есть но добавить нормировку по жаккарду 
- посчитать как есть но смотреть по совпадению хотя бы одного
- найти новую метрику для мультилейбловой классификации
- посчитать Ф-меру
- близость к экспертной